In [12]:
import os
import pandas as pd



# [1] Cấu hình thư mục gốc 
DATA_DIR = "/data/raw"

# Nhóm Bảng chính (Main Tables)
MAIN_TRAIN_FILE = os.path.join(DATA_DIR, "application_train.csv")
MAIN_TEST_FILE  = os.path.join(DATA_DIR, "application_test.csv")

# Nhóm Luồng lịch sử BÊN NGOÀI Home Credit (Credit Bureau Stream)
BUREAU_FILE      = os.path.join(DATA_DIR, "bureau.csv")
BUREAU_BAL_FILE  = os.path.join(DATA_DIR, "bureau_balance.csv")

# Nhóm Luồng lịch sử BÊN TRONG Home Credit (Internal Behavioral Stream)
PREV_APP_FILE    = os.path.join(DATA_DIR, "previous_application.csv")
INS_PAYMENT_FILE = os.path.join(DATA_DIR, "installments_payments.csv")
POS_CASH_FILE    = os.path.join(DATA_DIR, "POS_CASH_balance.csv")
CREDIT_CARD_FILE = os.path.join(DATA_DIR, "credit_card_balance.csv")

# File phụ bổ sung thông tin mô tả
COL_DESC_FILE    = os.path.join(DATA_DIR, "HomeCredit_columns_description.csv")

In [13]:
def run_step3_isolated_inspection(file_variable_path):
    
    file_name = os.path.basename(file_variable_path)
    
    print(f"🚀 [Scan full]: {file_name}")
    print("-" * 65)
    
    if not os.path.exists(file_variable_path):
        print(f"❌ LỖI: Không tìm thấy file tại {file_variable_path}\n")
        return None
    
    df = pd.read_csv(file_variable_path)
    
    rows, cols = df.shape
    missing_by_col = df.isnull().sum()
    total_missing = missing_by_col.sum()
    dup_count = df.duplicated().sum()
    
    num_cols_count = len(df.select_dtypes(include=[np.number]).columns)
    cat_cols_count = len(df.select_dtypes(include=['object', 'category', 'string']).columns)
    
    dtype_counts = df.dtypes.value_counts()
    
    # In báo cáo trực quan ra màn hình Jupyter
    print(f"1️⃣ Kích thước ma trận: {rows:,} dòng | {cols} cột")    
    print(f"\n2️⃣ Phân loại thuộc tính (Data Types Summary):")
    print(f"   - Số lượng cột dạng SỐ  (Numerical):   {num_cols_count} cột")
    print(f"   - Số lượng cột dạng CHỮ (Categorical): {cat_cols_count} cột")
    print(f"   📊 Chi tiết định dạng hệ thống:")
    print(dtype_counts.to_string())
    
    print(f"\n3️⃣ Đánh giá ô trống: {total_missing:,} ô trên toàn bảng ({((total_missing)/(rows*cols))*100:.2f}%)")
    if total_missing > 0:
        # Tạo bảng thống kê khuyết dòng
        missing_df = pd.DataFrame({
            'Số ô trống': missing_by_col[missing_by_col > 0],
            'Tỷ lệ khuyết (%)': (missing_by_col[missing_by_col > 0] / rows) * 100
        }).sort_values(by='Tỷ lệ khuyết (%)', ascending=False)
        
        # LỌC CÁC CỘT CÓ TỶ LỆ KHUYẾT TỪ 30% TRỞ LÊN
        missing_30_df = missing_df[missing_df['Tỷ lệ khuyết (%)'] >= 30.0]
        
        print(f"   📌 Danh sách TOÀN BỘ cột khuyết từ 30% trở lên (Tổng cộng: {len(missing_30_df)} cột):")
        if not missing_30_df.empty:
            print(missing_30_df.to_string())
        else:
            print("   (Chúc mừng! Không có cột nào bị khuyết trên 30%)")
        
    print(f"\n4️⃣ Trạng thái trùng lặp 100%: {dup_count:,} dòng")
    
    print(f"\n5️⃣ Bản xem trước 3 dòng đầu của {file_name}:")
    display(df.head(3))
    print("=" * 65 + "\n")
    
    # Đóng gói kết quả bảo toàn vào Dictionary
    data_bundle = {
        'file_name': file_name,
        'df': df,
        'shape': (rows, cols),
        'num_cols_count': num_cols_count,
        'cat_cols_count': cat_cols_count,
        'missing_stats': missing_by_col,
        'duplicate_count': dup_count
    }
    return data_bundle

In [14]:
# cingChạy và cất toàn bộ dữ liệu tập Train vào biến 'res_main_train'
res_main_train = run_step3_isolated_inspection(MAIN_TRAIN_FILE)

🚀 [Scan full]: application_train.csv
-----------------------------------------------------------------
1️⃣ Kích thước ma trận: 307,511 dòng | 122 cột

2️⃣ Phân loại thuộc tính (Data Types Summary):
   - Số lượng cột dạng SỐ  (Numerical):   106 cột
   - Số lượng cột dạng CHỮ (Categorical): 16 cột
   📊 Chi tiết định dạng hệ thống:
float64    65
int64      41
str        16

3️⃣ Đánh giá ô trống: 9,152,465 ô trên toàn bảng (24.40%)
   📌 Danh sách TOÀN BỘ cột khuyết từ 30% trở lên (Tổng cộng: 50 cột):
                              Số ô trống  Tỷ lệ khuyết (%)
COMMONAREA_MEDI                   214865         69.872297
COMMONAREA_AVG                    214865         69.872297
COMMONAREA_MODE                   214865         69.872297
NONLIVINGAPARTMENTS_MEDI          213514         69.432963
NONLIVINGAPARTMENTS_MODE          213514         69.432963
NONLIVINGAPARTMENTS_AVG           213514         69.432963
FONDKAPREMONT_MODE                210295         68.386172
LIVINGAPARTMENTS_MODE     

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
res_bureau = run_step3_isolated_inspection(BUREAU_FILE)

🚀 [Scan full]: bureau.csv
-----------------------------------------------------------------
1️⃣ Kích thước ma trận: 1,716,428 dòng | 17 cột

2️⃣ Phân loại thuộc tính (Data Types Summary):
   - Số lượng cột dạng SỐ  (Numerical):   14 cột
   - Số lượng cột dạng CHỮ (Categorical): 3 cột
   📊 Chi tiết định dạng hệ thống:
float64    8
int64      6
str        3

3️⃣ Đánh giá ô trống: 3,939,947 ô trên toàn bảng (13.50%)
   📌 Danh sách TOÀN BỘ cột khuyết từ 30% trở lên (Tổng cộng: 4 cột):
                        Số ô trống  Tỷ lệ khuyết (%)
AMT_ANNUITY                1226791         71.473490
AMT_CREDIT_MAX_OVERDUE     1124488         65.513264
DAYS_ENDDATE_FACT           633653         36.916958
AMT_CREDIT_SUM_LIMIT        591780         34.477415

4️⃣ Trạng thái trùng lặp 100%: 0 dòng

5️⃣ Bản xem trước 3 dòng đầu của bureau.csv:


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN


In [16]:
res_bal_bureau = run_step3_isolated_inspection(BUREAU_BAL_FILE)

🚀 [Scan full]: bureau_balance.csv
-----------------------------------------------------------------
1️⃣ Kích thước ma trận: 27,299,925 dòng | 3 cột

2️⃣ Phân loại thuộc tính (Data Types Summary):
   - Số lượng cột dạng SỐ  (Numerical):   2 cột
   - Số lượng cột dạng CHỮ (Categorical): 1 cột
   📊 Chi tiết định dạng hệ thống:
int64    2
str      1

3️⃣ Đánh giá ô trống: 0 ô trên toàn bảng (0.00%)

4️⃣ Trạng thái trùng lặp 100%: 0 dòng

5️⃣ Bản xem trước 3 dòng đầu của bureau_balance.csv:


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C


res_prev_app = run_step3_isolated_inspection(PREV_APP_FILE)

In [17]:
res_ins_payment = run_step3_isolated_inspection(INS_PAYMENT_FILE)

🚀 [Scan full]: installments_payments.csv
-----------------------------------------------------------------
1️⃣ Kích thước ma trận: 13,605,401 dòng | 8 cột

2️⃣ Phân loại thuộc tính (Data Types Summary):
   - Số lượng cột dạng SỐ  (Numerical):   8 cột
   - Số lượng cột dạng CHỮ (Categorical): 0 cột
   📊 Chi tiết định dạng hệ thống:
float64    5
int64      3

3️⃣ Đánh giá ô trống: 5,810 ô trên toàn bảng (0.01%)
   📌 Danh sách TOÀN BỘ cột khuyết từ 30% trở lên (Tổng cộng: 0 cột):
   (Chúc mừng! Không có cột nào bị khuyết trên 30%)

4️⃣ Trạng thái trùng lặp 100%: 0 dòng

5️⃣ Bản xem trước 3 dòng đầu của installments_payments.csv:


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000


In [18]:
res_pos_cash = run_step3_isolated_inspection(POS_CASH_FILE)

🚀 [Scan full]: POS_CASH_balance.csv
-----------------------------------------------------------------
1️⃣ Kích thước ma trận: 10,001,358 dòng | 8 cột

2️⃣ Phân loại thuộc tính (Data Types Summary):
   - Số lượng cột dạng SỐ  (Numerical):   7 cột
   - Số lượng cột dạng CHỮ (Categorical): 1 cột
   📊 Chi tiết định dạng hệ thống:
int64      5
float64    2
str        1

3️⃣ Đánh giá ô trống: 52,158 ô trên toàn bảng (0.07%)
   📌 Danh sách TOÀN BỘ cột khuyết từ 30% trở lên (Tổng cộng: 0 cột):
   (Chúc mừng! Không có cột nào bị khuyết trên 30%)

4️⃣ Trạng thái trùng lặp 100%: 0 dòng

5️⃣ Bản xem trước 3 dòng đầu của POS_CASH_balance.csv:


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0


In [19]:
res_credit_card = run_step3_isolated_inspection(CREDIT_CARD_FILE)

🚀 [Scan full]: credit_card_balance.csv
-----------------------------------------------------------------
1️⃣ Kích thước ma trận: 3,840,312 dòng | 23 cột

2️⃣ Phân loại thuộc tính (Data Types Summary):
   - Số lượng cột dạng SỐ  (Numerical):   22 cột
   - Số lượng cột dạng CHỮ (Categorical): 1 cột
   📊 Chi tiết định dạng hệ thống:
float64    15
int64       7
str         1

3️⃣ Đánh giá ô trống: 5,877,356 ô trên toàn bảng (6.65%)
   📌 Danh sách TOÀN BỘ cột khuyết từ 30% trở lên (Tổng cộng: 0 cột):
   (Chúc mừng! Không có cột nào bị khuyết trên 30%)

4️⃣ Trạng thái trùng lặp 100%: 0 dòng

5️⃣ Bản xem trước 3 dòng đầu của credit_card_balance.csv:


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0


In [21]:
import os
import pandas as pd

def run_one_click_missing_override():
    # 🎯 Đường dẫn file của bạn
    COL_DESC_FILE = "/notebook/DataDescription.csv"
    DATA_DIR = "/data/raw"

    # Ánh xạ cấu trúc bảng
    file_mapping = {
        'application_{train|test}.csv': os.path.join(DATA_DIR, "application_train.csv"),  
        'bureau.csv': os.path.join(DATA_DIR, "bureau.csv"),
        'bureau_balance.csv': os.path.join(DATA_DIR, "bureau_balance.csv"),
        'previous_application.csv': os.path.join(DATA_DIR, "previous_application.csv"),
        'installments_payments.csv': os.path.join(DATA_DIR, "installments_payments.csv"),
        'POS_CASH_balance.csv': os.path.join(DATA_DIR, "POS_CASH_balance.csv"),
        'credit_card_balance.csv': os.path.join(DATA_DIR, "credit_card_balance.csv")
    }
    
    print("🔄 [Khởi động]: Đang đọc file cấu trúc mô tả từ Notion...")
    if not os.path.exists(COL_DESC_FILE):
        print(f"❌ LỖI: Không tìm thấy file tại đường dẫn: {COL_DESC_FILE}")
        return
        
    # Đọc file (sử dụng utf-8 hoặc latin1 tùy thuộc định dạng xuất của Notion)
    try:
        df_desc = pd.read_csv(COL_DESC_FILE, encoding='utf-8')
    except Exception:
        df_desc = pd.read_csv(COL_DESC_FILE, encoding='latin1')
        
    # 🛠 SỬA LỖI NOTION: Làm sạch khoảng trắng và chuẩn hóa tên cột
    df_desc.columns = [str(c).strip() for c in df_desc.columns]
    print(f"📊 Các cột tìm thấy trong file Notion của bạn hiện tại: {list(df_desc.columns)}")
    
    # Tự động dò tìm cột đóng vai trò là 'Row' (Tên cột) và 'Table' (Tên bảng)
    row_col_name = None
    table_col_name = None
    
    for c in df_desc.columns:
        if c.lower() in ['row', 'feature', 'variable', 'tên cột', 'property']:
            row_col_name = c
        if c.lower() in ['table', 'file', 'tên bảng', 'bảng']:
            table_col_name = c
            
    # Nếu không dò ra tên mặc định, ép buộc lấy cột số 2 và số 3 làm Row và Table (đặc trưng cấu trúc file mô tả)
    if not row_col_name:
        row_col_name = 'Row' if 'Row' in df_desc.columns else df_desc.columns[2]
    if not table_col_name:
        table_col_name = 'Table' if 'Table' in df_desc.columns else df_desc.columns[1]
        
    print(f"🎯 Hệ thống tự động xác định: Cột biến là '{row_col_name}' | Cột bảng là '{table_col_name}'")
    
    # Làm sạch dữ liệu trong cột để khớp biểu thức
    df_desc[row_col_name] = df_desc[row_col_name].astype(str).str.strip()
    df_desc[table_col_name] = df_desc[table_col_name].astype(str).str.strip()
    
    # Khởi tạo hoặc đặt lại giá trị mặc định cho 2 cột mới
    df_desc['Missing_Percentage'] = "0.00%"
    df_desc['Missing_Category'] = "Dưới 30%"
    
    # Quét qua từng file để tính toán bằng kỹ thuật Chunking
    for table_pattern, actual_file_path in file_mapping.items():
        if not os.path.exists(actual_file_path):
            print(f"⚠️ Bỏ qua: Không tìm thấy file thực tế tại {actual_file_path}")
            continue
            
        print(f"🚀 Đang quét và tính tỷ lệ khuyết cho: {os.path.basename(actual_file_path)}")
        
        # 1. Đọc nhanh lấy danh sách tên cột
        col_names = pd.read_csv(actual_file_path, nrows=0).columns
        missing_counts = {col: 0 for col in col_names}
        total_rows = 0
        
        # 2. Đọc theo từng cụm (Chunking) tránh tràn RAM
        chunk_size = 100000
        for chunk in pd.read_csv(actual_file_path, chunksize=chunk_size):
            total_rows += len(chunk)
            null_counts = chunk.isnull().sum()
            for col in col_names:
                missing_counts[col] += null_counts[col]
                
        # 3. Tính toán tỷ lệ phần trăm và gán ngược
        for col_name, m_count in missing_counts.items():
            missing_pct = (m_count / total_rows) * 100
            
            if missing_pct > 70.0:
                category = "Trên 70%"
            elif 30.0 <= missing_pct <= 70.0:
                category = "Từ 30% đến 70%"
            else:
                category = "Dưới 30%"
                
            pct_str = f"{missing_pct:.2f}%"
            
            # Khớp vị trí theo tên cột động vừa tìm được
            condition = (df_desc[row_col_name] == col_name) & (df_desc[table_col_name] == table_pattern)
            if condition.any():
                df_desc.loc[condition, 'Missing_Percentage'] = pct_str
                df_desc.loc[condition, 'Missing_Category'] = category

    # 4. Lưu đè trực tiếp lên file gốc
    try:
        df_desc.to_csv(COL_DESC_FILE, index=False)
        print("\n" + "="*65)
        print(f"✨ HOÀN THÀNH 100%! Đã xử lý cấu trúc Notion và ghi đè thành công:\n👉 {COL_DESC_FILE}")
        print("="*65)
    except PermissionError:
        print("\n❌ THẤT BẠI: Vui lòng TẮT ứng dụng Excel/Numbers đang mở file này trước khi chạy lại!")

# Kích hoạt chạy lại hàm
run_one_click_missing_override()

🔄 [Khởi động]: Đang đọc file cấu trúc mô tả từ Notion...
📊 Các cột tìm thấy trong file Notion của bạn hiện tại: ['Row', 'ID', 'Table', 'Description', 'Phân loại chiến lược', 'Lý do chi tiết', 'Special', 'Missing_Rate', 'Data_Type', 'Unique_Count', 'IV_Score', 'Feature_Importance_Baseline']
🎯 Hệ thống tự động xác định: Cột biến là 'Row' | Cột bảng là 'Table'
🚀 Đang quét và tính tỷ lệ khuyết cho: application_train.csv
🚀 Đang quét và tính tỷ lệ khuyết cho: bureau.csv
🚀 Đang quét và tính tỷ lệ khuyết cho: bureau_balance.csv
🚀 Đang quét và tính tỷ lệ khuyết cho: previous_application.csv
🚀 Đang quét và tính tỷ lệ khuyết cho: installments_payments.csv
🚀 Đang quét và tính tỷ lệ khuyết cho: POS_CASH_balance.csv
🚀 Đang quét và tính tỷ lệ khuyết cho: credit_card_balance.csv

✨ HOÀN THÀNH 100%! Đã xử lý cấu trúc Notion và ghi đè thành công:
👉 /Users/nguyenminhtri/FinalYearPro/notebook/DataDescription.csv
